# Staging Data Validation

This notebook validates the Walmart M5 source CSV files against the SQL Server staging layer.It does not insert, update, delete, truncate, or alter SQL Server tables.

It validates:
- `calendar.csv` - `staging.stg_calendar`
- `sell_prices.csv` - `staging.stg_sell_prices`
- `sales_train_validation.csv` - `staging.stg_sales`
- sales unpivot reconciliation using the same `units_sold > 0` rule as the ETL
- NULLs, duplicates, key integrity, value checks, and source-to-staging row counts

In [21]:
# Install if needed:
# %pip install pandas sqlalchemy pyodbc

## 1. Configuration

In [22]:
from pathlib import Path
import urllib
import pandas as pd
from sqlalchemy import create_engine, text

BASE_PATH = Path(r"C:\Users\janen\OneDrive\Desktop\walmart")

CALENDAR_CSV = BASE_PATH / "calendar.csv"
PRICES_CSV = BASE_PATH / "sell_prices.csv"
SALES_CSV = BASE_PATH / "sales_train_validation.csv"

SERVER = r"JANEY-JESHEN"
DATABASE = "WalmartBI"
SCHEMA = "staging"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;")

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect="
    + urllib.parse.quote_plus(connection_string),
    pool_pre_ping=True)

print(f"Server   : {SERVER}")
print(f"Database : {DATABASE}")

Server   : JANEY-JESHEN
Database : WalmartBI


## 2. Helper functions

In [23]:
validation_results = []

def record_check(check, expected, actual, passed, details=""):
    validation_results.append({
        "check": check,
        "expected": expected,
        "actual": actual,
        "status": "PASS" if passed else "FAIL",
        "details": details
    })

def check_equal(check, expected, actual, details=""):
    passed = expected == actual
    record_check(check, expected, actual, passed, details)
    return passed

def check_true(check, condition, expected=True, actual=None, details=""):
    if actual is None:
        actual = condition
    record_check(check, expected, actual, bool(condition), details)
    return bool(condition)

def sql_scalar(query, params=None):
    with engine.connect() as connection:
        return connection.execute(text(query), params or {}).scalar()

def sql_df(query, params=None):
    with engine.connect() as connection:
        return pd.read_sql(text(query), connection, params=params or {})

def table_count(table):
    return sql_scalar(f"SELECT COUNT_BIG(*) FROM {SCHEMA}.{table}")

def section(title):
    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)

## 3. Source file checks

In [24]:
section("SOURCE FILE CHECK")

source_files = {"calendar.csv": CALENDAR_CSV,"sell_prices.csv": PRICES_CSV,
                    "sales_train_validation.csv": SALES_CSV}

for name, path in source_files.items():
    exists = path.exists()
    print(f"{name:30} : {'FOUND' if exists else 'MISSING'}")
    check_true(f"Source file exists: {name}", exists, details=str(path))

if not all(p.exists() for p in source_files.values()):
    raise FileNotFoundError("Fix the source paths in Section 1 before continuing.")


----------------------------------------------------------------------
SOURCE FILE CHECK
----------------------------------------------------------------------
calendar.csv                   : FOUND
sell_prices.csv                : FOUND
sales_train_validation.csv     : FOUND


## 4. Calendar source validation

In [25]:
section("CALENDAR — SOURCE VALIDATION")

calendar = pd.read_csv(CALENDAR_CSV)

expected_calendar_columns = [
    "date", "wm_yr_wk", "weekday", "wday", "month", "year", "d",
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "snap_CA", "snap_TX", "snap_WI"
]

print("Shape:", calendar.shape)
print("Columns:", calendar.columns.tolist())

check_equal("Calendar rows", 1969, len(calendar))
check_equal("Calendar columns", 14, len(calendar.columns))
check_equal("Calendar schema", expected_calendar_columns, calendar.columns.tolist())
check_equal("Calendar duplicate rows", 0, int(calendar.duplicated().sum()))

required = [
    "date", "wm_yr_wk", "weekday", "wday", "month", "year", "d",
    "snap_CA", "snap_TX", "snap_WI"
]
check_equal("Calendar required-field NULLs", 0, int(calendar[required].isna().sum().sum()))

calendar["date"] = pd.to_datetime(calendar["date"], errors="coerce")
check_equal("Calendar invalid dates", 0, int(calendar["date"].isna().sum()))
check_equal("Calendar unique d values", 1969, calendar["d"].nunique())

display(calendar.isna().sum().to_frame("missing_count"))


----------------------------------------------------------------------
CALENDAR — SOURCE VALIDATION
----------------------------------------------------------------------
Shape: (1969, 14)
Columns: ['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']


,missing_count
date,0
wm_yr_wk,0
weekday,0
wday,0
month,0
year,0
d,0
event_name_1,1807
event_type_1,1807
event_name_2,1964


## 5. Sell-price source validation

In [26]:
section("SELL PRICES — SOURCE VALIDATION")

prices = pd.read_csv(PRICES_CSV)

expected_price_columns = ["store_id", "item_id", "wm_yr_wk", "sell_price"]

print("Shape:", prices.shape)
print("Columns:", prices.columns.tolist())

check_equal("Price rows", 6_841_121, len(prices))
check_equal("Price columns", 4, len(prices.columns))
check_equal("Price schema", expected_price_columns, prices.columns.tolist())
check_equal("Price duplicate rows", 0, int(prices.duplicated().sum()))

prices["wm_yr_wk"] = pd.to_numeric(prices["wm_yr_wk"], errors="coerce")
prices["sell_price"] = pd.to_numeric(prices["sell_price"], errors="coerce")

check_equal(
    "Price required-field NULLs", 0,
    int(prices[["store_id", "item_id", "wm_yr_wk"]].isna().sum().sum())
)
check_equal("Invalid price week", 0, int(prices["wm_yr_wk"].isna().sum()))
check_equal("Invalid sell price", 0, int(prices["sell_price"].isna().sum()))
check_equal("Negative prices", 0, int((prices["sell_price"] < 0).sum()))

business_key_dupes = int(
    prices.duplicated(["store_id", "item_id", "wm_yr_wk"]).sum()
)
check_equal("Price business-key duplicates", 0, business_key_dupes)

display(prices.isna().sum().to_frame("missing_count"))


----------------------------------------------------------------------
SELL PRICES — SOURCE VALIDATION
----------------------------------------------------------------------
Shape: (6841121, 4)
Columns: ['store_id', 'item_id', 'wm_yr_wk', 'sell_price']


,missing_count
store_id,0
item_id,0
wm_yr_wk,0
sell_price,0


## 6. Sales source validation

In [27]:
section("SALES — SOURCE VALIDATION")

sales = pd.read_csv(SALES_CSV)

id_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
daily_columns = [c for c in sales.columns if c.startswith("d_")]

print("Shape:", sales.shape)
print("Identifier columns:", len(id_columns))
print("Daily columns:", len(daily_columns))

check_equal("Sales source rows", 30_490, len(sales))
check_equal("Sales daily columns", 1_913, len(daily_columns))
check_equal("Sales total columns", 1_919, len(sales.columns))
check_equal("Sales raw duplicate rows", 0, int(sales.duplicated().sum()))
check_equal("Sales ID-key duplicates", 0, int(sales.duplicated(id_columns).sum()))
check_equal("Sales identifier NULLs", 0, int(sales[id_columns].isna().sum().sum()))

missing_ids = [c for c in id_columns if c not in sales.columns]
check_equal("Missing sales identifier columns", [], missing_ids)


----------------------------------------------------------------------
SALES — SOURCE VALIDATION
----------------------------------------------------------------------
Shape: (30490, 1919)
Identifier columns: 6
Daily columns: 1913


True

In [28]:
section("SALES — OBSERVATION VALIDATION")

# Theoretical number of product-day cells
possible_observations = len(sales) * len(daily_columns)

# Actual non-NULL sales observations
actual_observations = int(
    sales[daily_columns].notna().sum().sum()
)

print("Possible observations:", possible_observations)
print("Actual non-NULL observations:", actual_observations)

check_equal(
    "Possible sales observations",
    possible_observations,
    actual_observations
)


----------------------------------------------------------------------
SALES — OBSERVATION VALIDATION
----------------------------------------------------------------------
Possible observations: 58327370
Actual non-NULL observations: 58327370


True

## 7. Calculating the exact expected sales staging count

In [8]:
section("SALES — ETL RECONCILIATION")

possible_observations = len(sales) * len(daily_columns)

# Same logical filter as the ETL:
# melted = melted[melted["units_sold"] > 0]
positive_sales_count = int(sales[daily_columns].gt(0).sum().sum())
non_positive_count = int(sales[daily_columns].le(0).sum().sum())
null_sales_count = int(sales[daily_columns].isna().sum().sum())

print(f"Possible daily observations : {possible_observations:,}")
print(f"Positive sales observations : {positive_sales_count:,}")
print(f"Zero/negative observations  : {non_positive_count:,}")
print(f"NULL sales observations     : {null_sales_count:,}")

check_equal("Possible sales observations", 58_328_370, possible_observations)
check_equal(
    "Sales observation decomposition",
    possible_observations,
    positive_sales_count + non_positive_count + null_sales_count
)


----------------------------------------------------------------------
SALES — ETL RECONCILIATION
----------------------------------------------------------------------
Possible daily observations : 58,327,370
Positive sales observations : 18,550,276
Zero/negative observations  : 39,777,094
NULL sales observations     : 0


True

## 8. SQL Server connection

In [9]:
section("SQL SERVER CONNECTION")

server_name = sql_scalar("SELECT @@SERVERNAME")
database_name = sql_scalar("SELECT DB_NAME()")

print("Server  :", server_name)
print("Database:", database_name)

check_equal("Connected database", DATABASE, database_name)


----------------------------------------------------------------------
SQL SERVER CONNECTION
----------------------------------------------------------------------
Server  : Janey-jeshen
Database: WalmartBI


True

## 9. Staging table existence

In [10]:
section("STAGING TABLE EXISTENCE")

staging_tables = ["stg_calendar", "stg_sell_prices", "stg_sales"]

for table in staging_tables:
    exists = sql_scalar(
        """
        SELECT COUNT(*)
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = :schema_name
          AND TABLE_NAME = :table_name
        """,
        {"schema_name": SCHEMA, "table_name": table}
    ) == 1

    print(f"{SCHEMA}.{table:20} : {'FOUND' if exists else 'MISSING'}")
    check_true(f"Table exists: {SCHEMA}.{table}", exists)


----------------------------------------------------------------------
STAGING TABLE EXISTENCE
----------------------------------------------------------------------
staging.stg_calendar         : FOUND
staging.stg_sell_prices      : FOUND
staging.stg_sales            : FOUND


## 10. Source vs staging row-count reconciliation

In [11]:
section("SOURCE VS STAGING")

staging_calendar_count = table_count("stg_calendar")
staging_prices_count = table_count("stg_sell_prices")
staging_sales_count = table_count("stg_sales")

reconciliation = pd.DataFrame({
    "dataset": ["calendar", "sell_prices", "sales"],
    "expected_staging_rows": [
        len(calendar),
        len(prices),
        positive_sales_count
    ],
    "actual_staging_rows": [
        staging_calendar_count,
        staging_prices_count,
        staging_sales_count
    ]
})

reconciliation["difference"] = (
    reconciliation["actual_staging_rows"]
    - reconciliation["expected_staging_rows"]
)
reconciliation["status"] = reconciliation["difference"].eq(0).map(
    {True: "PASS", False: "FAIL"}
)

display(reconciliation)

check_equal("Calendar source vs staging", len(calendar), staging_calendar_count)
check_equal("Prices source vs staging", len(prices), staging_prices_count)
check_equal("Sales expected vs staging", positive_sales_count, staging_sales_count)


----------------------------------------------------------------------
SOURCE VS STAGING
----------------------------------------------------------------------


,dataset,expected_staging_rows,actual_staging_rows,difference,status
0,calendar,1969,1969,0,PASS
1,sell_prices,6841121,6841121,0,PASS
2,sales,18550276,18550276,0,PASS


True

## 11. Calendar staging quality

In [12]:
section("STG_CALENDAR — DATA QUALITY")

calendar_quality = sql_df("""
SELECT
    COUNT_BIG(*) AS total_rows,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS date_nulls,
    SUM(CASE WHEN wm_yr_wk IS NULL THEN 1 ELSE 0 END) AS week_nulls,
    SUM(CASE WHEN d IS NULL THEN 1 ELSE 0 END) AS d_nulls,
    SUM(CASE WHEN snap_CA IS NULL THEN 1 ELSE 0 END) AS snap_ca_nulls,
    SUM(CASE WHEN snap_TX IS NULL THEN 1 ELSE 0 END) AS snap_tx_nulls,
    SUM(CASE WHEN snap_WI IS NULL THEN 1 ELSE 0 END) AS snap_wi_nulls
FROM staging.stg_calendar
""")
display(calendar_quality)

duplicate_d = sql_scalar("""
SELECT COUNT(*)
FROM (
    SELECT d
    FROM staging.stg_calendar
    GROUP BY d
    HAVING COUNT(*) > 1
) x
""")
check_equal("Calendar duplicate d keys", 0, duplicate_d)

bad_snap = sql_scalar("""
SELECT COUNT(*)
FROM staging.stg_calendar
WHERE snap_CA NOT IN (0,1)
   OR snap_TX NOT IN (0,1)
   OR snap_WI NOT IN (0,1)
""")
check_equal("Invalid SNAP rows", 0, bad_snap)


----------------------------------------------------------------------
STG_CALENDAR — DATA QUALITY
----------------------------------------------------------------------


,total_rows,date_nulls,week_nulls,d_nulls,snap_ca_nulls,snap_tx_nulls,snap_wi_nulls
0,1969,0,0,0,0,0,0


True

## 12. Sell-price staging quality

In [13]:
section("STG_SELL_PRICES — DATA QUALITY")

price_quality = sql_df("""
SELECT
    COUNT_BIG(*) AS total_rows,
    SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) AS store_nulls,
    SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) AS item_nulls,
    SUM(CASE WHEN wm_yr_wk IS NULL THEN 1 ELSE 0 END) AS week_nulls,
    SUM(CASE WHEN sell_price IS NULL THEN 1 ELSE 0 END) AS price_nulls,
    SUM(CASE WHEN sell_price < 0 THEN 1 ELSE 0 END) AS negative_prices
FROM staging.stg_sell_prices
""")
display(price_quality)

duplicate_price_keys = sql_scalar("""
SELECT COUNT(*)
FROM (
    SELECT store_id, item_id, wm_yr_wk
    FROM staging.stg_sell_prices
    GROUP BY store_id, item_id, wm_yr_wk
    HAVING COUNT(*) > 1
) x
""")
check_equal("Price duplicate business keys", 0, duplicate_price_keys)


----------------------------------------------------------------------
STG_SELL_PRICES — DATA QUALITY
----------------------------------------------------------------------


,total_rows,store_nulls,item_nulls,week_nulls,price_nulls,negative_prices
0,6841121,0,0,0,0,0


True

## 13. Sales staging quality

In [14]:
section("STG_SALES — DATA QUALITY")

sales_quality = sql_df("""
SELECT
    COUNT_BIG(*) AS total_rows,
    SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS id_nulls,
    SUM(CASE WHEN item_id IS NULL THEN 1 ELSE 0 END) AS item_nulls,
    SUM(CASE WHEN dept_id IS NULL THEN 1 ELSE 0 END) AS dept_nulls,
    SUM(CASE WHEN cat_id IS NULL THEN 1 ELSE 0 END) AS cat_nulls,
    SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END) AS store_nulls,
    SUM(CASE WHEN state_id IS NULL THEN 1 ELSE 0 END) AS state_nulls,
    SUM(CASE WHEN d IS NULL THEN 1 ELSE 0 END) AS d_nulls,
    SUM(CASE WHEN units_sold IS NULL THEN 1 ELSE 0 END) AS units_nulls,
    SUM(CASE WHEN units_sold <= 0 THEN 1 ELSE 0 END) AS non_positive_sales
FROM staging.stg_sales
""")
display(sales_quality)

duplicate_sales_keys = sql_scalar("""
SELECT COUNT(*)
FROM (
    SELECT id, d
    FROM staging.stg_sales
    GROUP BY id, d
    HAVING COUNT(*) > 1
) x
""")
check_equal("Sales duplicate id+d keys", 0, duplicate_sales_keys)

non_positive = sql_scalar("""
SELECT COUNT_BIG(*)
FROM staging.stg_sales
WHERE units_sold <= 0
""")
check_equal("Non-positive sales rows", 0, non_positive)


----------------------------------------------------------------------
STG_SALES — DATA QUALITY
----------------------------------------------------------------------


,total_rows,id_nulls,item_nulls,dept_nulls,cat_nulls,store_nulls,state_nulls,d_nulls,units_nulls,non_positive_sales
0,18550276,0,0,0,0,0,0,0,0,0


True

## 14. Sales - calendar referential check

In [15]:
section("SALES → CALENDAR REFERENTIAL CHECK")

missing_days = sql_scalar("""
SELECT COUNT(*)
FROM (
    SELECT DISTINCT s.d
    FROM staging.stg_sales s
    LEFT JOIN staging.stg_calendar c
        ON s.d = c.d
    WHERE c.d IS NULL
) x
""")

print("Sales day keys missing from calendar:", missing_days)
check_equal("Sales day keys missing from calendar", 0, missing_days)


----------------------------------------------------------------------
SALES → CALENDAR REFERENTIAL CHECK
----------------------------------------------------------------------
Sales day keys missing from calendar: 0


True

## 15. Inspect staging schemas

In [16]:
section("STAGING SCHEMAS")

for table in staging_tables:
    print(f"\n{SCHEMA}.{table}")
    schema_df = sql_df("""
        SELECT
            ORDINAL_POSITION,
            COLUMN_NAME,
            DATA_TYPE,
            CHARACTER_MAXIMUM_LENGTH,
            IS_NULLABLE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = :schema_name
          AND TABLE_NAME = :table_name
        ORDER BY ORDINAL_POSITION
    """, {"schema_name": SCHEMA, "table_name": table})
    display(schema_df)


----------------------------------------------------------------------
STAGING SCHEMAS
----------------------------------------------------------------------

staging.stg_calendar


,ORDINAL_POSITION,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,IS_NULLABLE
0,1,date,date,NaN,NO
1,2,wm_yr_wk,smallint,NaN,NO
2,3,weekday,nvarchar,50.0,NO
3,4,wday,tinyint,NaN,NO
4,5,month,tinyint,NaN,NO
5,6,year,smallint,NaN,NO
6,7,d,nvarchar,50.0,NO
7,8,event_name_1,nvarchar,50.0,YES
8,9,event_type_1,nvarchar,50.0,YES
9,10,event_name_2,nvarchar,50.0,YES



staging.stg_sell_prices


,ORDINAL_POSITION,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,IS_NULLABLE
0,1,store_id,nvarchar,50.0,NO
1,2,item_id,nvarchar,50.0,NO
2,3,wm_yr_wk,smallint,NaN,NO
3,4,sell_price,float,NaN,NO



staging.stg_sales


,ORDINAL_POSITION,COLUMN_NAME,DATA_TYPE,CHARACTER_MAXIMUM_LENGTH,IS_NULLABLE
0,1,id,varchar,50.0,YES
1,2,item_id,varchar,50.0,YES
2,3,dept_id,varchar,50.0,YES
3,4,cat_id,varchar,50.0,YES
4,5,store_id,varchar,20.0,YES
5,6,state_id,varchar,10.0,YES
6,7,d,varchar,10.0,YES
7,8,units_sold,int,NaN,YES


## 16. Final validation summary

In [17]:
section("FINAL VALIDATION SUMMARY")

results_df = pd.DataFrame(validation_results)
display(results_df)

passed = int((results_df["status"] == "PASS").sum())
failed = int((results_df["status"] == "FAIL").sum())

print(f"Checks executed : {len(results_df)}")
print(f"Passed          : {passed}")
print(f"Failed          : {failed}")

if failed == 0:
    print("\nFINAL RESULT: PASS — staging data passed all validation checks.")
else:
    print("\nFINAL RESULT: FAIL — review the failed checks above.")


----------------------------------------------------------------------
FINAL VALIDATION SUMMARY
----------------------------------------------------------------------


,check,expected,actual,status,details
0,Source file exists: calendar.csv,True,True,PASS,C:\Users\janen\OneDrive\Desktop\walmart\calend...
1,Source file exists: sell_prices.csv,True,True,PASS,C:\Users\janen\OneDrive\Desktop\walmart\sell_p...
2,Source file exists: sales_train_validation.csv,True,True,PASS,C:\Users\janen\OneDrive\Desktop\walmart\sales_...
3,Calendar rows,1969,1969,PASS,
4,Calendar columns,14,14,PASS,
5,Calendar schema,"[date, wm_yr_wk, weekday, wday, month, year, d...","[date, wm_yr_wk, weekday, wday, month, year, d...",PASS,
6,Calendar duplicate rows,0,0,PASS,
7,Calendar required-field NULLs,0,0,PASS,
8,Calendar invalid dates,0,0,PASS,
9,Calendar unique d values,1969,1969,PASS,


Checks executed : 41
Passed          : 40
Failed          : 1

FINAL RESULT: FAIL — review the failed checks above.


In [18]:
daily_columns = [
    col for col in sales.columns
    if col.startswith("d_")
]

print("Sales rows:", len(sales))
print("Daily columns:", len(daily_columns))
print("Theoretical observations:", len(sales) * len(daily_columns))

Sales rows: 30490
Daily columns: 1913
Theoretical observations: 58327370


## 17. Portfolio reconciliation report


In [19]:
report = pd.DataFrame({
    "Dataset": ["Calendar", "Sell Prices", "Sales"],
    "Raw Source Rows": [len(calendar), len(prices), len(sales)],
    "Transformation": [
        "Clean/load",
        "Clean/load",
        "Unpivot d_1..d_1913 + keep units_sold > 0"
    ],
    "Expected Staging Rows": [
        len(calendar),
        len(prices),
        positive_sales_count
    ],
    "Actual Staging Rows": [
        staging_calendar_count,
        staging_prices_count,
        staging_sales_count
    ]
})

report["Difference"] = (
    report["Actual Staging Rows"]
    - report["Expected Staging Rows"]
)
report["Status"] = report["Difference"].eq(0).map(
    {True: "PASS", False: "FAIL"}
)

display(report)

print(
    "\nOVERALL RECONCILIATION:",
    "PASS" if report["Status"].eq("PASS").all() else "FAIL"
)

,Dataset,Raw Source Rows,Transformation,Expected Staging Rows,Actual Staging Rows,Difference,Status
0,Calendar,1969,Clean/load,1969,1969,0,PASS
1,Sell Prices,6841121,Clean/load,6841121,6841121,0,PASS
2,Sales,30490,Unpivot d_1..d_1913 + keep units_sold > 0,18550276,18550276,0,PASS



OVERALL RECONCILIATION: PASS


## Expected source shapes for this project

Based on the source files you inspected:

- `calendar.csv`: **(1969, 14)**
- `sell_prices.csv`: **(6841121, 4)**
- `sales_train_validation.csv`: **(30490, 1919)**

For sales:

`30,490 × 1,913 = 58,328,370` possible daily observations.

The notebook calculates the actual number of positive observations from the CSV file instead of hard-coding the final `stg_sales` count.